In [52]:
import pandas as pd

df = pd.read_csv("C:/Users/munur/Documents/GitHub/B-13-RealtyAI-Smart-Real-Estate-Insight-Platform/datasets/real_estate_dataset.csv")
df.head()

,url,beds,city,date,size,type,baths,price,neighborhood
0,https://www.99acres.com/residential-land-plot-...,0,Bangalore,2025-02-19,799-1258 sqft,Residential land / Plot,0,2317000.0,Jigani
1,https://www.99acres.com/2-bhk-bedroom-apartmen...,2,Bangalore,2025-02-19,1085 sqft,2 BHK Flat,2,12500000.0,Tumkur Road
2,https://www.99acres.com/sumadhura-capitol-resi...,3,Bangalore,2025-02-19,1525-2150 sqft,"3, 4 BHK Apartment",0,23200000.0,Whitefield
3,https://www.99acres.com/provident-botanico-sou...,2,Bangalore,2025-02-19,658-1003 sqft,"2, 3 BHK Apartment",0,889000.0,Soukya Road
4,https://www.99acres.com/kvg-superior-kalkere-b...,2,Bangalore,2025-02-19,1179-1449 sqft,"2, 3 BHK Apartment",0,6483000.0,Kalkere


In [53]:
print(df.columns)

Index(['url', 'beds', 'city', 'date', 'size', 'type', 'baths', 'price',
       'neighborhood'],
      dtype='str')


1.url - DROP(Unique ID, no relation to price)
2.beds - No encoding(Numeric quantity)
3.baths - Numeric quantity
4.size - Convert range → average sqft 
5.type - (converted Type into preoperty Type and BHK no.)
6.price - No encoding (Value you predict)
7.city - one hot encoding (only 26 different cities)
8.neighborhood - Frequency Encoding (224 different places)
9.date - not sure

In [54]:
df.drop(columns=['url'], inplace=True)
df.head()

,beds,city,date,size,type,baths,price,neighborhood
0,0,Bangalore,2025-02-19,799-1258 sqft,Residential land / Plot,0,2317000.0,Jigani
1,2,Bangalore,2025-02-19,1085 sqft,2 BHK Flat,2,12500000.0,Tumkur Road
2,3,Bangalore,2025-02-19,1525-2150 sqft,"3, 4 BHK Apartment",0,23200000.0,Whitefield
3,2,Bangalore,2025-02-19,658-1003 sqft,"2, 3 BHK Apartment",0,889000.0,Soukya Road
4,2,Bangalore,2025-02-19,1179-1449 sqft,"2, 3 BHK Apartment",0,6483000.0,Kalkere


In [55]:
import re

def extract_bhk(x):
    match = re.search(r'\d+', str(x))
    if match:
        return int(match.group())
    return None

df['bhk'] = df['type'].apply(extract_bhk)

In [56]:
def extract_property(x):
    x = str(x).lower()

    if 'plot' in x or 'land' in x:
        return 'plot'
    elif 'apartment' in x:
        return 'apartment'
    elif 'flat' in x:
        return 'flat'
    else:
        return 'other'

df['property_type'] = df['type'].apply(extract_property)

In [57]:
df.head()

,beds,city,date,size,type,baths,price,neighborhood,bhk,property_type
0,0,Bangalore,2025-02-19,799-1258 sqft,Residential land / Plot,0,2317000.0,Jigani,NaN,plot
1,2,Bangalore,2025-02-19,1085 sqft,2 BHK Flat,2,12500000.0,Tumkur Road,2.0,flat
2,3,Bangalore,2025-02-19,1525-2150 sqft,"3, 4 BHK Apartment",0,23200000.0,Whitefield,3.0,apartment
3,2,Bangalore,2025-02-19,658-1003 sqft,"2, 3 BHK Apartment",0,889000.0,Soukya Road,2.0,apartment
4,2,Bangalore,2025-02-19,1179-1449 sqft,"2, 3 BHK Apartment",0,6483000.0,Kalkere,2.0,apartment


In [58]:
df.drop(columns=['type'], inplace=True)
df.head()

,beds,city,date,size,baths,price,neighborhood,bhk,property_type
0,0,Bangalore,2025-02-19,799-1258 sqft,0,2317000.0,Jigani,NaN,plot
1,2,Bangalore,2025-02-19,1085 sqft,2,12500000.0,Tumkur Road,2.0,flat
2,3,Bangalore,2025-02-19,1525-2150 sqft,0,23200000.0,Whitefield,3.0,apartment
3,2,Bangalore,2025-02-19,658-1003 sqft,0,889000.0,Soukya Road,2.0,apartment
4,2,Bangalore,2025-02-19,1179-1449 sqft,0,6483000.0,Kalkere,2.0,apartment


In [59]:
import re
import numpy as np

def convert_sqft(x):
    if pd.isna(x):
        return np.nan

    # make lowercase string
    x = str(x).lower()

    # remove sqft and commas
    x = x.replace('sqft', '')
    x = x.replace(',', '').strip()

    # case 1: range values
    if '-' in x:
        parts = x.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan

    # case 2: single value
    try:
        return float(x)
    except:
        return np.nan


df['size'] = df['size'].apply(convert_sqft)

In [60]:
df.head()

,beds,city,date,size,baths,price,neighborhood,bhk,property_type
0,0,Bangalore,2025-02-19,1028.5,0,2317000.0,Jigani,NaN,plot
1,2,Bangalore,2025-02-19,1085.0,2,12500000.0,Tumkur Road,2.0,flat
2,3,Bangalore,2025-02-19,1837.5,0,23200000.0,Whitefield,3.0,apartment
3,2,Bangalore,2025-02-19,830.5,0,889000.0,Soukya Road,2.0,apartment
4,2,Bangalore,2025-02-19,1314.0,0,6483000.0,Kalkere,2.0,apartment


In [61]:
df['city'] = df['city'].str.strip().str.title()

In [62]:
df = pd.get_dummies(df, columns=['city'], drop_first=True, dtype=int)

In [63]:
df.head()

,beds,date,size,baths,price,neighborhood,bhk,property_type,city_Bangalore,city_Bhubaneswar,...,city_Mumbai,city_Navi Mumbai,city_Noida,city_Patna,city_Pune,city_Thane,city_Vijayawada,city_Vikarabad,city_Visakhapatnam,city_Vizianagaram
0,0,2025-02-19,1028.5,0,2317000.0,Jigani,NaN,plot,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,2025-02-19,1085.0,2,12500000.0,Tumkur Road,2.0,flat,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,2025-02-19,1837.5,0,23200000.0,Whitefield,3.0,apartment,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2,2025-02-19,830.5,0,889000.0,Soukya Road,2.0,apartment,1,0,...,0,0,0,0,0,0,0,0,0,0
4,2,2025-02-19,1314.0,0,6483000.0,Kalkere,2.0,apartment,1,0,...,0,0,0,0,0,0,0,0,0,0


In [64]:
df['neighborhood'] = df['neighborhood'].str.strip().str.title()

In [65]:
mean_price = df.groupby('neighborhood')['price'].mean()

df['neighborhood'] = df['neighborhood'].map(mean_price)

In [66]:
df.head()

,beds,date,size,baths,price,neighborhood,bhk,property_type,city_Bangalore,city_Bhubaneswar,...,city_Mumbai,city_Navi Mumbai,city_Noida,city_Patna,city_Pune,city_Thane,city_Vijayawada,city_Vikarabad,city_Visakhapatnam,city_Vizianagaram
0,0,2025-02-19,1028.5,0,2317000.0,2317000.0,NaN,plot,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,2025-02-19,1085.0,2,12500000.0,12500000.0,2.0,flat,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,2025-02-19,1837.5,0,23200000.0,17150000.0,3.0,apartment,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2,2025-02-19,830.5,0,889000.0,889000.0,2.0,apartment,1,0,...,0,0,0,0,0,0,0,0,0,0
4,2,2025-02-19,1314.0,0,6483000.0,6483000.0,2.0,apartment,1,0,...,0,0,0,0,0,0,0,0,0,0
